# VAE Latent Space — Interactive UMAP Dashboard

Encodes all cells in a split, runs UMAP, then:
- Displays an interactive Plotly scatter **in this cell** (one trace per condition)
- Saves a standalone HTML dashboard to `OUT_DIR/umap_dashboard.html` that replicates the kymograph-viewer style: click a point → see membrane + nuclei side by side; box-select → grid of cells.

In [1]:
import sys, io, json, base64
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import umap
from PIL import Image
import plotly.graph_objects as go
from IPython.display import display, HTML

try:
    from tqdm.notebook import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

sys.path.insert(0, str(Path("..").resolve() / "src"))
from darth_vaeder.models import LitVAE
from darth_vaeder.datamodules import MultinucDataModule

In [2]:
# ── config ────────────────────────────────────────────────────────────────────
CKPT      = "/mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/checkpoints/version_16/epoch=36-val_loss=285.1519.ckpt"
ZARR      = "/mnt/efs/dl_jrc/student_data/S-JS/multinucleation.zarr"
TABLE     = "/mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/outputs/cell_table.csv"
OUT_DIR   = Path("outputs/umap_explorer")
SPLIT     = "test"       # "train" | "val" | "test"
MAX_CELLS = None         # None = all cells; e.g. 3000 for faster iteration
THUMB_PX  = 80           # pixels per channel in thumbnail (side-by-side → 80×160 total)

# UMAP
N_NEIGHBORS = 15
MIN_DIST    = 0.1
SEED        = 42

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output  : {OUT_DIR.resolve()}")

Output  : /mnt/efs/dl_jrc/student_data/S-JS/repos/Darth_VAEder/Joaquin'scripts/outputs/umap_explorer


In [3]:
# ── model ─────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = LitVAE.load_from_checkpoint(CKPT, map_location=device)
model.eval()
print(f"z_dim={model.hparams.z_dim}  nc={model.hparams.nc}  device={device}")

z_dim=32  nc=4  device=cuda


In [4]:
# ── datamodule ────────────────────────────────────────────────────────────────
dm = MultinucDataModule(
    data_path=ZARR,
    cell_table_csv=TABLE,
    channels=(0, 1),
    batch_size=256,
    num_workers=4,
    augment=False,
    pin_memory=False,
    persistent_workers=False,
)
dm.setup("fit")
dm.setup("test")

loader_map = {
    "train": dm.train_dataloader,
    "val":   dm.val_dataloader,
    "test":  dm.test_dataloader,
}
loader = loader_map[SPLIT]()
print(f"Split '{SPLIT}': {len(loader.dataset)} cells")

Split 'test': 1666 cells


In [5]:
# ── thumbnail helper ──────────────────────────────────────────────────────────
def make_thumbnail(img: np.ndarray, size: int = THUMB_PX) -> str:
    """(2, H, W) float32 normalised → base64 PNG, channels side by side."""
    panels = []
    for ch in range(img.shape[0]):
        arr = np.clip(img[ch], 0.0, 1.0)
        pil = Image.fromarray((arr * 255).astype(np.uint8), mode="L")
        pil = pil.resize((size, size), Image.LANCZOS)
        panels.append(np.array(pil))
    composite = np.concatenate(panels, axis=1)   # (size, 2*size)
    buf = io.BytesIO()
    Image.fromarray(composite, mode="L").save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

In [7]:
# ── encode cells + generate thumbnails ───────────────────────────────────────
all_mu, all_idx, all_thumbs = [], [], []
all_condition, all_replicate, all_imgname, all_localidx = [], [], [], []

with torch.no_grad():
    for batch in tqdm(loader, desc="encoding"):
        x_img    = batch[model.hparams.image_key].to(device)
        mask     = batch[model.hparams.mask_key].to(device)
        nuc_mask = batch[model.hparams.nuc_mask_key].to(device)
        x_in     = torch.cat([x_img, mask.float(), nuc_mask.float()], dim=1)
        _, _, mu, _ = model.vae(x_in)

        all_mu.append(mu.cpu().numpy())
        all_idx.extend(batch["index"].tolist())

        imgs_np = batch[model.hparams.image_key].cpu().numpy()  # (B, 2, H, W)
        for i in range(len(imgs_np)):
            all_thumbs.append(make_thumbnail(imgs_np[i]))

        meta = batch["metadata"]
        all_condition.extend(meta["condition"])
        all_replicate.extend(meta["replicate"])
        all_imgname.extend(meta["image_name"])
        all_localidx.extend([int(v) for v in meta["local_cell_index"]])

mu_all = np.concatenate(all_mu, axis=0)   # (N, z_dim)
print(f"Encoded {len(mu_all)} cells  |  mu shape: {mu_all.shape}")

# optional subsampling
if MAX_CELLS and len(mu_all) > MAX_CELLS:
    rng  = np.random.default_rng(SEED)
    keep = rng.choice(len(mu_all), MAX_CELLS, replace=False)
    keep = np.sort(keep)
    mu_all        = mu_all[keep]
    all_idx       = [all_idx[i]       for i in keep]
    all_thumbs    = [all_thumbs[i]    for i in keep]
    all_condition = [all_condition[i] for i in keep]
    all_replicate = [all_replicate[i] for i in keep]
    all_imgname   = [all_imgname[i]   for i in keep]
    all_localidx  = [all_localidx[i]  for i in keep]
    print(f"Subsampled to {len(mu_all)} cells")

encoding:   0%|          | 0/7 [00:00<?, ?it/s]

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x4096 and 576x64)

In [8]:
# ── UMAP ──────────────────────────────────────────────────────────────────────
reducer = umap.UMAP(
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    n_components=2,
    random_state=SEED,
    verbose=True,
)
xy = reducer.fit_transform(mu_all)   # (N, 2)
print(f"UMAP done: {xy.shape}")

/home/S-JS/conda/envs/darth-vaeder/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(n_jobs=1, random_state=42, verbose=True)
Mon Jun 15 21:30:42 2026 Construct fuzzy simplicial set
Mon Jun 15 21:30:45 2026 Finding Nearest Neighbors
Mon Jun 15 21:30:50 2026 Finished Nearest Neighbor Search
Mon Jun 15 21:30:54 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Mon Jun 15 21:30:57 2026 Finished embedding
UMAP done: (1666, 2)


In [9]:
# ── build shared data structure ───────────────────────────────────────────────
conditions = list(dict.fromkeys(all_condition))   # preserve order, unique

# items list: one entry per cell (global index into this array)
items = [
    {
        "condition": all_condition[i],
        "replicate": str(all_replicate[i]),
        "filename":  f"{all_replicate[i]}|{all_condition[i]}|{all_imgname[i]}|cell_{all_localidx[i]}",
        "thumbnail": all_thumbs[i],
    }
    for i in range(len(mu_all))
]

# classes: one per condition, referencing global item indices
cond_to_global = {c: [] for c in conditions}
for i, c in enumerate(all_condition):
    cond_to_global[c].append(i)

classes = [
    {
        "name":    cond,
        "x":       xy[cond_to_global[cond], 0].tolist(),
        "y":       xy[cond_to_global[cond], 1].tolist(),
        "indices": cond_to_global[cond],
    }
    for cond in conditions
]

data_dict = {
    "z_dim":   int(model.hparams.z_dim),
    "split":   SPLIT,
    "n_cells": len(items),
    "classes": classes,
    "items":   items,
}
print(f"Conditions: {conditions}")
print(f"Cells per condition: { {c: len(cond_to_global[c]) for c in conditions} }")

Conditions: ['MATURE', 'CTRL', 'CMs25d']
Cells per condition: {'MATURE': 628, 'CTRL': 342, 'CMs25d': 696}


In [10]:
# ── interactive Plotly scatter (notebook cell) ─────────────────────────────────
PAL = [
    "#4e9de0", "#ff8c42", "#e040fb", "#d62728", "#2ca02c",
    "#9467bd", "#8c564b", "#e377c2", "#bcbd22", "#17becf",
]

fig = go.Figure()
for i, cls in enumerate(classes):
    fig.add_trace(go.Scatter(
        x=cls["x"],
        y=cls["y"],
        mode="markers",
        name=cls["name"],
        marker=dict(size=5, opacity=0.85, color=PAL[i % len(PAL)]),
        customdata=cls["indices"],
        hovertemplate=(
            f"<b>{cls['name']}</b><br>"
            "global_idx=%{customdata}<extra></extra>"
        ),
    ))

fig.update_layout(
    title=f"UMAP  (z_dim={model.hparams.z_dim}, {SPLIT} split)",
    height=700,
    clickmode="event+select",
    dragmode="pan",
    margin=dict(l=50, r=20, t=60, b=50),
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(title="UMAP 1", gridcolor="#eee", zeroline=False),
    yaxis=dict(title="UMAP 2", gridcolor="#eee", zeroline=False),
    legend=dict(bgcolor="white", bordercolor="#ddd", borderwidth=1),
)
fig.show()

In [11]:
# ── build + save standalone HTML dashboard ────────────────────────────────────
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <title>VAE – Latent Space Explorer</title>
  <script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
  <style>
    *, *::before, *::after { box-sizing: border-box }
    body  { font-family: Arial, sans-serif; margin: 0; padding: 16px;
            background: #f0f0f0; color: #222 }
    h2    { margin: 0 0 4px }
    .sub  { color: #999; font-size: 13px; margin: 0 0 14px }
    #wrap      { display: flex; gap: 16px; align-items: flex-start }
    #plot-col  { flex: 3 }
    #panel-col { flex: 2; min-width: 420px; max-height: 760px;
                 overflow-y: auto; background: #fff; border-radius: 8px;
                 padding: 14px; box-shadow: 0 2px 8px #bbb }
    #panel-col h4 { margin: 0 0 10px; font-size: 15px }
    .hint { color: #ccc; font-style: italic }
    .ibox b    { font-size: 14px }
    .ibox .meta { font-size: 11px; color: #999; word-break: break-all; margin: 2px 0 0 }
    .img-bg { background: #0a0a0a; padding: 4px; border-radius: 4px;
               width: 100%; margin-top: 6px }
    .img-bg img { width: 100%; display: block; image-rendering: pixelated }
    .ch-labels { display: flex; width: 100%; border-bottom: 1px solid #eee; margin-top: 2px }
    .ch-labels span { flex: 1; text-align: center; font-size: 10px; color: #999; padding: 2px 0 }
    .ghdr { font-size: 13px; font-weight: bold; margin-bottom: 8px }
    .grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 6px }
    .tile { background: #111; border-radius: 3px; padding: 3px 3px 0 3px }
    .tile img  { width: 100%; display: block; image-rendering: pixelated }
    .tile .tlbl { font-size: 9px; color: #bbb; text-align: center; padding: 2px 0 3px }
  </style>
</head>
<body>
<h2>VAE Latent Space Explorer</h2>
<p class="sub">
  Click a point to view its cell image &nbsp;&middot;&nbsp;
  Box/lasso-select for a grid view &nbsp;&middot;&nbsp;
  Scroll to zoom &nbsp;&middot;&nbsp; Drag to pan<br>
  <small>Left panel: <b>membrane</b> &nbsp;|&nbsp; right panel: <b>nuclei</b>
  (both normalised to [0,1])</small>
</p>
<div id="wrap">
  <div id="plot-col"><div id="umap-plot"></div></div>
  <div id="panel-col">
    <h4>Cell Image Viewer</h4>
    <div id="panel-content">
      <p class="hint">&#x1F446; Click a point or box-select a region.</p>
    </div>
  </div>
</div>
<script>
const DATA = __DATA_JSON__;
const PAL = [
  "#4e9de0","#ff8c42","#e040fb","#d62728","#2ca02c",
  "#9467bd","#8c564b","#e377c2","#bcbd22","#17becf"
];
const traces = DATA.classes.map((cls, i) => ({
  x:          cls.x,
  y:          cls.y,
  mode:       "markers",
  type:       "scatter",
  name:       cls.name,
  customdata: cls.indices,
  marker:     { size: 5, opacity: 0.85, color: PAL[i % PAL.length] },
  hovertemplate: "<b>" + cls.name + "</b><br>idx=%{customdata}<extra></extra>",
}));
Plotly.newPlot("umap-plot", traces, {
  title:         { text: "UMAP  (z_dim=" + DATA.z_dim + ", " + DATA.split + " split)", font: { size: 16 } },
  height:        720,
  clickmode:     "event+select",
  dragmode:      "pan",
  margin:        { l: 50, r: 20, t: 50, b: 50 },
  plot_bgcolor:  "#fff",
  paper_bgcolor: "#fff",
  xaxis:         { title: "UMAP 1", gridcolor: "#eee" },
  yaxis:         { title: "UMAP 2", gridcolor: "#eee" },
  legend:        { bgcolor: "#fff", bordercolor: "#ddd", borderwidth: 1 },
}, { scrollZoom: true, responsive: true });
const panel = document.getElementById("panel-content");
function singleView(globalIdx) {
  const it = DATA.items[globalIdx];
  panel.innerHTML =
    `<div class="ibox">
       <b>${it.condition} &mdash; ${it.replicate}</b>
       <div class="meta">${it.filename}</div>
     </div>
     <div class="img-bg">
       <img src="${it.thumbnail}" title="${it.condition} | ${it.replicate}">
     </div>
     <div class="ch-labels"><span>membrane</span><span>nuclei</span></div>`;
}
function gridView(pts) {
  const MAX = 12, n = pts.length;
  let h =
    `<div class="ghdr">${n} point${n !== 1 ? "s" : ""} selected` +
    `${n > MAX ? " — first " + MAX + " shown" : ""}</div>` +
    `<div class="grid">`;
  for (let i = 0; i < Math.min(n, MAX); i++) {
    const it = DATA.items[pts[i].customdata];
    h += `<div class="tile">
            <img src="${it.thumbnail}" title="${it.filename}">
            <div class="tlbl">${it.condition} | ${it.replicate}</div>
          </div>`;
  }
  panel.innerHTML = h + `</div>`;
}
const el = document.getElementById("umap-plot");
el.on("plotly_click",    ev => singleView(ev.points[0].customdata));
el.on("plotly_selected", ev => { if (ev && ev.points.length) gridView(ev.points); });
</script>
</body>
</html>"""

# embed data and write
data_json  = json.dumps(data_dict, separators=(",", ":"))
html_out   = HTML_TEMPLATE.replace("__DATA_JSON__", data_json)
out_path   = OUT_DIR / "umap_dashboard.html"
out_path.write_text(html_out, encoding="utf-8")

size_mb = out_path.stat().st_size / 1e6
print(f"Saved: {out_path}  ({size_mb:.1f} MB)")
print(f"Open in a browser — click any point to inspect its membrane + nuclei channels.")

Saved: outputs/umap_explorer/umap_dashboard.html  (3.1 MB)
Open in a browser — click any point to inspect its membrane + nuclei channels.
